## About this notebook

This notebook segments files selected for calculating alignment in each round.

Input:
- pathway to a directory containing selected files for the segmentation (im_to_segment)
- list of wells to process

Output:
- segmented masks of selected channels

## Choose the experiment

Set `CONFIG` to the experiment's `experiment.json`. All folders and the well list come from it; create or edit it with `setup_experiment.ipynb`.

In [ ]:
from fouri import load_config

# This experiment's experiment.json (create or edit it with setup_experiment.ipynb)
CONFIG = r'Z:\PATH\TO\EXPERIMENT\experiment.json'

exp = load_config(CONFIG)

data_path = exp.output_images  # images selected for segmentation (output of 00)
save_path = exp.segmented      # where to save segmented images
wells = exp.well_list

# expected diameter of objects for segmentation in pixels (default 30)
selected_diameter = 30

## Prepare for segmentation

In [ ]:
import os

from cellpose import models
from tifffile import imread
from tqdm.notebook import tqdm

from fouri import make_dir, segment_image, write_tiff


In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        f"No GPU available to PyTorch (torch {torch.__version__}, CUDA build: {torch.version.cuda}) "
        "If CUDA build is None, CPU only torch was installed. Run `uv sync`."
    )

print(
    f"GPU: {torch.cuda.get_device_name(0)}"
)

torch.sparse.check_sparse_tensor_invariants.disable()

In [ ]:
make_dir(save_path)

In [ ]:
model = models.CellposeModel(gpu=True)
if model.device.type != "cuda":
    raise RuntimeError(f"Cellpose is running on {model.device}, not the GPU.")

## Process selected wells

In [ ]:
# loop for segmentation

well_bar = tqdm(wells, desc="Overall Progress", unit="well")
for well in well_bar:

    files = sorted(f for f in os.listdir(os.path.join(data_path, well)) if f.endswith('.tif'))

    make_dir(os.path.join(save_path,well))

    img_bar = tqdm(files, desc=f"Segmenting {well}", unit="round", leave=False)
    for i, img_name in enumerate(img_bar):

        # specify pathway to the image
        img_path = os.path.join(data_path,well,img_name)

        # get an image
        im = imread(img_path)

        # segment the right plane
        round_num = int(os.path.splitext(img_name)[0].rsplit("_", 1)[-1])
        labels, peak_gb = segment_image(model, im, selected_diameter, desc=f"Segmenting round {round_num}")

        # save segmentation
        path_save_temp = os.path.join(save_path,well,img_name.replace('im2segment','labels'))
        write_tiff(path_save_temp, labels)

        if peak_gb is not None:
            img_bar.set_postfix(peak_VRAM=f"{peak_gb:.2f}GB")